In [1]:
# !pip install "numpy<2"

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as lt
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

/home/codespace/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/codespace/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (
/home/codespace/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
df = pd.read_parquet('./data/yellow_tripdata_2023-01.parquet')

**Q1: Number of Columns**


In [4]:
num_columns = len(df.columns)
print(f'number of columns: {num_columns}')

number of columns: 19


**Q2: Computing duration**

In [5]:
# Calculating the duration first and convert to minutes
def duration_creation(df):
    df['duration'] = df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
    df['duration'] = df['duration'].dt.total_seconds() / 60
    return df
df = duration_creation(df)
std = df['duration'].std()
print(f"standard deviation: {std:.02f}")

standard deviation: 42.59


**Q3:  Dropping outliers**

In [6]:
def boxplot(column):
    sns.boxplot(column)
    lt.title('Boxplot of BMI')
    lt.show()
    
record_before = len(df['duration'])
df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]
# boxplot(df['duration'])

In [7]:
records_after = len(df['duration'])
fraction_left = int(np.round((records_after / record_before) * 100))
print(f"fraction left: {fraction_left}%")

fraction left: 98%


**Q4: One-hot encoding**

In [8]:
def one_hot(df, categorical_list):
    categorical_feature = categorical_list
    # numrical = ['passenger_count', 'trip_distance', 'duration']
    numrical = []
    df[categorical_feature] = df[categorical_feature].astype(str)
    new_df = df[categorical_feature + numrical]
    return new_df
new_df = one_hot(df, ['PULocationID', 'DOLocationID'])
dv = DictVectorizer()
df_dict = new_df.to_dict(orient="records")
X_train = dv.fit_transform(df_dict)

In [9]:
print(f"Number of Columns: {X_train.shape[1]}")

Number of Columns: 515


**Q5: Training a model**

In [10]:
Y_train = df['duration'].values

lr = LinearRegression()
lr.fit(X_train, Y_train)
       

y_pred = lr.predict(X_train)
rmse = mean_squared_error(Y_train, y_pred, squared=False)
print(f"RMSE: {rmse:.2f}")

RMSE: 7.65


**Q6: Evaluating the model**

In [11]:
print("read validation file...")
df_val = pd.read_parquet('./data/yellow_tripdata_2023-02.parquet')
print("Duratoin created for validation")
df_val = duration_creation(df_val)
df_val = df_val[(df_val['duration'] >= 1) & (df_val['duration'] <= 60)]
print("One hot encode...")
new_df_val = one_hot(df_val, ['PULocationID', 'DOLocationID'])
df_val_dict = new_df_val.to_dict(orient="records")
print("Begin vectorizing validation...")
X_val = dv.transform(df_val_dict)
Y_val = df_val['duration'].values
val_pred = lr.predict(X_val)
print("Predict the Validation")
rmse_val = mean_squared_error(Y_val, val_pred, squared=False)
print(f"RMSE Validation: {rmse_val:0.2f}")

read validation file...
Duratoin created for validation
One hot encode...
Begin vectorizing validation...
Predict the Validation
RMSE Validation: 13.31
